# Data Preprocessing 

### starting a trial preprocessing

In [2]:
import pandas as pd
import json
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

- load the data and interpretation 

In [3]:
df= pd.read_sas('../dataset/raw/Data.XPT', format='xport')
with open("../dataset/Document/selected_brfss_codebook.json", "r", encoding="utf-8") as f:
    codebook = json.load(f)

### apply some methods

In [4]:
df.shape

(433323, 350)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 433323 entries, 0 to 433322
Columns: 350 entries, _STATE to _DRNKDRV
dtypes: float64(345), object(5)
memory usage: 1.1+ GB


In [6]:
# Columns selected for stroke prediction
selected_columns = [
    'CVDSTRK3',
    '_AGE80',
    'SEXVAR',
    '_BMI5',
    '_RFHYPE6',
    'BPMEDS',
    'DIABETE4',
    'SMOKE100',
    '_SMOKER3',
    '_MICHD',
    'CVDINFR4',
    'CVDCRHD4',
    'TOLDHI3',
    'CHOLMED3',
    'CHCKDNY2',
    'PREDIAB2',
    'EXERANY2',
    '_TOTINDA',
    '_PAINDX3',
    'PAMIN13_',
    '_PA30023',
    'GENHLTH',
    'PHYSHLTH',
    'MENTHLTH',
    'EDUCA',
    'INCOME3',
    'EMPLOY1',
    'MARITAL'
]

# Keep only columns that actually exist in the dataset
available_columns = [
    col for col in selected_columns
    if col in df.columns
]

# Create filtered dataframe
df_selected = df[available_columns].copy()

print("Columns retained:")
print(df_selected.columns.tolist())

print("\nShape:")
print(df_selected.shape)

Columns retained:
['CVDSTRK3', '_AGE80', 'SEXVAR', '_BMI5', '_RFHYPE6', 'DIABETE4', 'SMOKE100', '_SMOKER3', '_MICHD', 'CVDINFR4', 'CVDCRHD4', 'TOLDHI3', 'CHOLMED3', 'CHCKDNY2', 'PREDIAB2', 'EXERANY2', '_TOTINDA', '_PAINDX3', 'PAMIN13_', '_PA30023', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'EDUCA', 'INCOME3', 'EMPLOY1', 'MARITAL']

Shape:
(433323, 27)


In [7]:
missing_codes = {
    'CVDSTRK3': [7, 9],
    '_RFHYPE6': [7, 9],
    'DIABETE4': [7, 9],
    'SMOKE100': [7, 9],
    '_SMOKER3': [9],
    '_MICHD': [9],
    'CVDINFR4': [7, 9],
    'CVDCRHD4': [7, 9],
    'TOLDHI3': [7, 9],
    'CHOLMED3': [7, 9],
    'CHCKDNY2': [7, 9],
    'PREDIAB2': [7, 9],
    'EXERANY2': [7, 9],
    '_PAINDX3': [9],
    'GENHLTH': [7, 9],
    'EDUCA': [9],
    'INCOME3': [77, 99],
    'EMPLOY1': [9],
    'MARITAL': [9]
}
import numpy as np

df_clean = df_selected.copy()

for col, codes in missing_codes.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace(codes, np.nan)

In [9]:
def decode_brfss(df, codebook, categorical_columns):
    """
    Converts BRFSS numeric codes into their actual meanings.

    Example:
        1 -> Male
        2 -> Female

    Numeric variables such as Age/BMI are left untouched.
    """

    df = df.copy()

    for col in categorical_columns:

        if col not in df.columns or col not in codebook:
            continue

        mapping = {}

        for item in codebook[col].get("values", []):
            code = str(item["code"]).strip()
            meaning = item["meaning"].strip()
            mapping[code] = meaning

        def decode(value):

            if pd.isna(value):
                return value

            # Convert 1.0 -> 1
            if isinstance(value, float) and value.is_integer():
                value = int(value)

            return mapping.get(str(value), value)

        df[col] = df[col].apply(decode)

    return df


In [10]:
def encode_features(df, categorical_columns, ordinal_columns=None):
    """
    Encodes categorical variables for ML.

    - Ordinal variables -> OrdinalEncoder
    - Nominal variables -> OneHotEncoder

    Numeric columns remain unchanged.
    """

    df = df.copy()

    ordinal_columns = ordinal_columns or []

    categorical_columns = [
        col for col in categorical_columns
        if col in df.columns
    ]

    nominal_columns = [
        col for col in categorical_columns
        if col not in ordinal_columns
    ]

    # --------------------------------------------------------
    # Ordinal encoding
    # --------------------------------------------------------

    if ordinal_columns:

        ordinal_encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )

        df[ordinal_columns] = ordinal_encoder.fit_transform(
            df[ordinal_columns]
        )

    # --------------------------------------------------------
    # One-hot encoding
    # --------------------------------------------------------

    if nominal_columns:

        df = pd.get_dummies(
            df,
            columns=nominal_columns,
            dtype=int
        )

    return df


# ============================================================
# 5. COMPLETE PIPELINE
# ============================================================

def prepare_dataset(
    df,
    codebook_path="selected_brfss_codebook.json",
    categorical_columns=None,
    ordinal_columns=None
):
    """
    Complete BRFSS preprocessing pipeline:

        Raw data
          ↓
        Decode
          ↓
        Clean special values
          ↓
        Encode
          ↓
        ML-ready dataframe
    """

    codebook 

    # 1. Decode
    df_decoded = decode_brfss(
        df,
        codebook,
        categorical_columns
    )

    # 2. Clean
    df_clean =(
        df_decoded
    )

    # 3. Encode
    df_encoded = encode_features(
        df_clean,
        categorical_columns,
        ordinal_columns
    )

    return df_encoded

In [ ]:
missing_report = pd.DataFrame({
    'Column': df_clean.columns,
    'Missing_Count': df_clean.isna().sum().values,
    'Missing_Percentage': (
        df_clean.isna().mean().values * 100
    ).round(2),
    'Non_Missing_Count': df_clean.notna().sum().values,
    'Data_Type': df_clean.dtypes.astype(str).values
})

missing_report = missing_report.sort_values(
    'Missing_Percentage',
    ascending=False
).reset_index(drop=True)

print(missing_report.to_string(index=False))


  Column  Missing_Count  Missing_Percentage  Non_Missing_Count Data_Type
PREDIAB2         268133               61.88             165190   float64
PAMIN13_         127612               29.45             305711   float64
 INCOME3          86623               19.99             346700   float64
_PAINDX3          60375               13.93             372948   float64
CHOLMED3          55651               12.84             377672   float64
 TOLDHI3          55084               12.71             378239   float64
   _BMI5          40535                9.35             392788   float64
_SMOKER3          23062                5.32             410261   float64
SMOKE100          22568                5.21             410755   float64
 EMPLOY1           7681                1.77             425642   float64
  _MICHD           4585                1.06             428738   float64
 MARITAL           4289                0.99             429034   float64
CVDCRHD4           4231                0.98        

In [25]:
# how to find total number of unique rows like no dulicarry one row only one time with missing values in the entire dataset
unique_rows_with_missing = df_clean[df_clean.isna().any(axis=1)].drop_duplicates().shape[0]

In [26]:
unique_rows_with_missing

362454

### we will not delete all the rows with missing values we will just handel it diffrently 


In [28]:
# Drop duplicate rows based on all columns, keeping the first occurrence
df_cleaned = df_clean.drop_duplicates(keep='first')


In [31]:
df_cleaned.shape

(433009, 27)

In [29]:
df_Clean = df_cleaned.dropna(subset=['CVDSTRK3']).copy()

print("Rows after removing missing target:", len(df_Clean))

Rows after removing missing target: 431540


In [ ]:
# for columns with a float values (Actual FLoat not some encoded float)
# but dont do it on real file before training testing split else data leakge will happen 
from sklearn.impute import SimpleImputer

numeric_cols = [
    '_AGE80',
    '_BMI5',
    'PAMIN13_',
    'PHYSHLTH',
    'MENTHLTH'
]

imputer = SimpleImputer(strategy='median')

df_Clean[numeric_cols] = imputer.fit_transform(
    df_Clean[numeric_cols]
)

In [ ]:
# dont drop instead drop columns
df_Clean = df_Clean.drop(columns=['PREDIAB2'])

In [40]:
# drop all rows with missing values in the entire dataset
df_Clean = df_Clean.dropna()